# Analysis: Run Summaries (vsv) → Aggregations & Charts

In diese Datei wurden die Besten Runs gespeichert.

Für interaktive und ausführliche Visualisierung siehe: https://wandb.ai/katja-gegg/multi-period-forecasting/workspace?nw=nwuserkatjagegg

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import json, os


In [ ]:

ROOT_DIR = Path.cwd().parents[1] if len(Path.cwd().parents) else Path.cwd()

EVAL_DIR = ROOT_DIR / "outputs/evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_SUMMARIES_PATH = EVAL_DIR / "runs_summary.csv"

path = r"outputs\evaluation\runs_summary.csv"

print("Evaluation dir:", EVAL_DIR.resolve())
print("Expecting run summaries at:", RUN_SUMMARIES_PATH.resolve())

In [ ]:
df_summaries = pd.read_csv(RUN_SUMMARIES_PATH) if RUN_SUMMARIES_PATH.exists() else pd.DataFrame()
df_summaries.sort_values(by=["Model","Datensatz", "RMSSE_total"], inplace=True)
df_summaries

In [ ]:
# ger csv from run summaries path

temp = df_summaries[["Run_Name", "Datensatz", "RMSSE_total", "Model"]].sort_values(by=["Datensatz", "RMSSE_total"], ascending=True)
# where Model = lgbm
temp = temp[temp["Model"] == "tft"]
temp


In [ ]:
# This code already gets the best (lowest) RMSSE per Model for the m5 dataset.
dataset = "m5"

# make a copy to avoid SettingWithCopyWarning
df_plot = df_summaries.copy()

df_plot = df_plot[df_plot["Datensatz"] == dataset].copy()
# sort by rmsse_total and only keep the best (lowest) per model
metric_col = "RMSSE_total" if "RMSSE_total" in df_plot.columns else "rmsse_total"
df_best_per_models = df_plot.loc[df_plot.groupby("Model")[metric_col].idxmin()].sort_values(by=metric_col)

# Show a compact table for reference
table = df_best_per_models[["Model", metric_col, "Run_Name", "WandB_URL"]].reset_index(drop=True)
table.rename(columns={metric_col: "RMSSE_total"}, inplace=True)
display(table)

# Plot
models = df_best_per_models["Model"].tolist()
values = df_best_per_models[metric_col].tolist()

# Let user select colors for each bar (example colors, replace as needed)
color_map = {
    'lgbm': '#2ca02c',   # green
    'xgb': '#1f77b4',    # blue
    'tft': '#d62728',    # red
    'nhits': '#ff7f0e',  # orange
}
custom_colors = [color_map.get(model, '#9467bd') for model in models]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, values, color=custom_colors)

# Annotate bars with values
for bar, val in zip(bars, values):
    # if nhits skip annotation here, will do it later
    if models[list(bars).index(bar)] != "nhits":
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{val:.3f}", ha="center", va="bottom")

# Add dotted black horizontal line at y=0.7
plt.axhline(0.7, color='black', linestyle=':', linewidth=2 )

# Move annotation for nhits bar (index 1) above the bar for readability
for i, (bar, val) in enumerate(zip(bars, values)):
    if models[i] == "nhits":
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
                 f"{val:.3f}", ha="center", va="bottom")
    else:
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f"{val:.3f}", ha="center", va="bottom")

plt.title(f"Best RMSSE_total pro Model — Datensatz: {dataset}")
plt.ylabel("RMSSE_total")
plt.xlabel("Model")
plt.tight_layout()

png_path = f"/mnt/data/best_rmsse_{dataset}.png"
plt.savefig(png_path, dpi=160, bbox_inches="tight")
plt.show()

png_path

In [ ]:
# get best of each dataset
df_best_by_dataset = df_summaries.loc[df_summaries.groupby("Datensatz")["rmsse_total"].idxmin()]
df_best_by_dataset

In [ ]:
# best by wrapper
df_best_by_wrapper = df_summaries.loc[df_summaries.groupby("Wrapper")["rmsse_total"].idxmin()]
df_best_by_wrapper

In [ ]:
# best by model
df_best_by_model = df_summaries.loc[df_summaries.groupby("Model")["rmsse_total"].idxmin()]
df_best_by_model

In [ ]:
# get best by model_klasse
df_best_by_model_klasse = df_summaries.loc[df_summaries.groupby("Model_Klasse")["rmsse_total"].idxmin()]
df_best_by_model_klasse

In [ ]:
# plots rmsse_total distributions
plt.figure(figsize=(10, 6))
plt.hist(df_summaries['rmsse_total'], bins=30, color='skyblue', edgecolor='black')
plt.title('Distribution of RMSSE Total')
plt.xlabel('RMSSE Total')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()
